In [21]:

import numpy as np
import sys
%load_ext autoreload
%autoreload 2
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 
from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
import matplotlib.pyplot as plt
from gauss_newton.problem import MultipleShooting
from gauss_newton.adaptive import run_optimization_adaptive
from scipy.interpolate import interp1d
from experiments.data_utils import  LogReaderV2, create_jax_interpolator
from mhe.mhe_base_model_interface import MheModel, MheCogeGenerator
from mhe.params import MheParams
from mhe.mhe_utils import MheEstimationData, run_mhe_estimation, plot_mhe_results, reset_mhe_solver, plot_mhe_data_windows
from commom_utils.system_config import create_system, create_mhe_params, SYSTEM_CONFIGS, MHE_CONFIGS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
root_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph")
# root_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Skipad/Left")

data_storage_EPS_angle = LogReaderV2(root_dir/"EPS_angle.csv")
data_storage_ESP_IMU = LogReaderV2(root_dir/"ESP_IMU.csv")
data_storage_IMU_2 = LogReaderV2(root_dir/"IMU_2.csv")
data_storage_wheel_speed = LogReaderV2(root_dir/"ESP_1_rear_speed_wheel.csv")

Loaded 28137 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph/EPS_angle.csv
Columns: ['time', 'EPS_SAS_SteerAngleValid', 'EPS_SAS_SteerAngleSpdValid', 'h144_undefine', 'h144_counter', 'h144_checksum']
Loaded 14070 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph/ESP_IMU.csv
Columns: ['time', 'LongAx', 'LattAx', 'YAW_Rate']
Loaded 9380 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph/IMU_2.csv
Columns: ['time', 'LongAx', 'LattAx', 'YAW_Rate']
Loaded 9380 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph/ESP_1_rear_speed_wheel.csv
Columns: ['time', 'IPB_wheelSpeedRL', 'IPB_wheelSpeedRR', 'h122_counter', 'h122_checksum']


In [3]:
print(data_storage_ESP_IMU.df_data.columns)
print(data_storage_EPS_angle.df_data.columns)
print(data_storage_wheel_speed.df_data.columns)
print(data_storage_IMU_2.df_data.columns)

Index(['time', 'LongAx', 'LattAx', 'YAW_Rate'], dtype='object')
Index(['time', 'EPS_SAS_SteerAngleValid', 'EPS_SAS_SteerAngleSpdValid',
       'h144_undefine', 'h144_counter', 'h144_checksum'],
      dtype='object')
Index(['time', 'IPB_wheelSpeedRL', 'IPB_wheelSpeedRR', 'h122_counter',
       'h122_checksum'],
      dtype='object')
Index(['time', 'LongAx', 'LattAx', 'YAW_Rate'], dtype='object')


In [4]:
data_storage_EPS_angle.add_batch( "EPS_SAS_SteerAngleValid", use_jax_interp=0, scale_coef = np.deg2rad(1))
data_storage_ESP_IMU.add_batch( "LattAx", use_jax_interp=0, scale_coef =-1)
data_storage_ESP_IMU.add_batch( "YAW_Rate", use_jax_interp=0,  scale_coef = -np.deg2rad(1))
data_storage_IMU_2.add_batch( "YAW_Rate", use_jax_interp=0,  scale_coef = -np.deg2rad(1))
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRL", use_jax_interp=0, scale_coef = 1.0/3.6)
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRR", use_jax_interp=0, scale_coef = 1.0/3.6)

data_storage_EPS_angle.process_all()
data_storage_ESP_IMU.process_all()
data_storage_IMU_2.process_all()
data_storage_wheel_speed.process_all()
#t = data_storage_EPS_angle.get_time("EPS_SAS_SteerAngleValid")
f_steer = data_storage_EPS_angle.get_f_interp("EPS_SAS_SteerAngleValid")
f_lat_ax = data_storage_ESP_IMU.get_f_interp("LattAx")
f_yaw_rate1 = data_storage_ESP_IMU.get_f_interp("YAW_Rate")
f_yaw_rate2 = data_storage_IMU_2.get_f_interp("YAW_Rate")
f_wheelSpeedRL = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRL")
f_wheelSpeedRR = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRR")


Queued EPS_SAS_SteerAngleValid: time [1780935294.714, 1780935388.495], values [-2.122, 5.695]
Queued LattAx: time [1780935294.717, 1780935388.497], values [-7.640, 7.840]
Queued YAW_Rate: time [1780935294.717, 1780935388.497], values [-0.512, 0.726]
Queued YAW_Rate: time [1780935294.718, 1780935388.497], values [-0.507, 0.729]
Queued IPB_wheelSpeedRL: time [1780935294.715, 1780935388.494], values [0.296, 17.733]
Queued IPB_wheelSpeedRR: time [1780935294.715, 1780935388.494], values [0.311, 17.562]

Set t0 = 1780935294.713968 (minimum of all start times)
Processed EPS_SAS_SteerAngleValid: original [1780935294.714, 1780935388.495], normalized [0.000, 93.781]
Common time range: [0.000, 93.781]

Set t0 = 1780935294.717373 (minimum of all start times)
Processed LattAx: original [1780935294.717, 1780935388.497], normalized [0.000, 93.780]
Processed YAW_Rate: original [1780935294.717, 1780935388.497], normalized [0.000, 93.780]
Common time range: [0.000, 93.780]

Set t0 = 1780935294.717603 (m

In [5]:
[data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2]

[np.float64(93.78077387809753),
 np.float64(93.77994012832642),
 np.float64(93.7797908782959)]

In [6]:
t2 = np.min([data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2])
t1 = np.max([data_storage_EPS_angle.t1, data_storage_ESP_IMU.t1, data_storage_IMU_2.t1])
t = np.arange(t1, t2 - 1, 0.02)

In [7]:
t

array([0.000e+00, 2.000e-02, 4.000e-02, ..., 9.272e+01, 9.274e+01,
       9.276e+01], shape=(4639,))

In [8]:
vx_data = (f_wheelSpeedRL(t) + f_wheelSpeedRR(t))/2
if(1):
    f_vx =  interp1d(t, vx_data, kind = 'linear') 
else:
    f_vx = create_jax_interpolator(t, vx_data, method='linear')

In [9]:
wheelbase = 2.96
gear_ratio_test  = 14.63
rwa = f_steer(t)/gear_ratio_test
vx = f_vx(t)
plt.plot(t,  vx*np.tan(rwa)/(wheelbase))
plt.plot(t, f_yaw_rate1(t))
#plt.plot(t, vx)
# plt.plot(t, f_steer(t))

# plt.plot(t, f_wheelSpeedRL(t))
# plt.plot(t, f_wheelSpeedRR(t))
# plt.plot(t, f_vx(t))
# plt.plot(t, f_yaw_rate1(t))
# plt.plot(t, f_yaw_rate2(t))

<Figure size 640x480 with 1 Axes>

In [10]:

plt.plot(t, vx*vx*rwa/wheelbase)
plt.plot(t, f_lat_ax(t))

<Figure size 640x480 with 1 Axes>

In [11]:
plt.plot(t, f_vx(t))

<Figure size 640x480 with 1 Axes>

In [12]:


def get_input_signals(t):
    steering = f_steer(t)
    Gr = 1.46367113e+01
    Gr_sq = 3.98385740e-02
    offset =  -3.02111275e-03
    rwa = (steering + offset)/(Gr - Gr_sq*(steering + offset)**2) 
    return [f_vx(t), rwa]      


class DynamicModelRearAxle(ODESystem):
    def __init__(self, m, wheelbase, g=9.81):
        self.m = m
        self.wheelbase = wheelbase
        self.g = g
        self.delay = DelaySystem(order=2)
        # состояния:  wz, vy_rear
        super().__init__(2, 5, 2)

    def get_lateral_forces(self, rwa, vx, vy_rear, wz, theta):
        Cf_norm, Cr_norm, a_rel = theta[0], theta[1], theta[2]
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        # Скорость центра масс через заднюю ось
        vy_cm = vy_rear + b * wz
        # alpha_f = rwa - (vy_cm + a * wz) / vx
        # alpha_r = -(vy_cm - b * wz) / vx   # = -vy_rear / vx

        alpha_f = rwa - ca.arctan((vy_cm + a * wz) / vx)
        alpha_r = ca.arctan(-vy_rear / vx)

        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr

    def get_derivative(self, state, params, u):
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        GR = params[4]
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # динамика центра масс (необходима для сил и ускорения)
        vy_cm = vy_rear + b * wz
        dvy_cm = (Fyf + Fyr) / self.m - vx * wz
        dwz = (a * Fyf - b * Fyr) / Iz

        # производная vy_rear
        dvy_rear = dvy_cm - b * dwz
        return ca.vertcat(dwz, dvy_rear)
    
    def calc_acc(self, state, params, u, d=0.0):
        """
        Вычисляет поперечное ускорение в точке, смещённой на d от центра масс.
        d > 0 – вперёд, к передней оси; d < 0 – назад.
        Состояние state (SX): [tau, psi, wz, vy_rear, rwa, rwa_dot, ...]
        """
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        GR = params[4]
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # Ускорение центра масс
        a_lat_cm = (Fyf + Fyr) / self.m

        # Угловое ускорение
        dwz = (a * Fyf - b * Fyr) / Iz

        # Ускорение в заданной точке
        a_lat = a_lat_cm + d * dwz

        return a_lat
    
    def observation(self, state: SX, theta: SX, u: SX):
        a_rel = theta[2]
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        a_lat = self.calc_acc(state, theta, u, d = -b)
        wz, vy = state[0], state[1]
        return ca.vertcat(a_lat, wz)

m = 2000

config_dyn = {
    "class": DynamicModelRearAxle,
    "args": [m, wheelbase],                                 # wheelbase
    "c0": np.array([0.0]),      
    "theta_true": np.array([2.88329956,  5.67880555,  0.48323357,  0.07280245, 12.47372793]),
    "delta_theta": np.array([0, 0, 0, 0, 0]),
    "input_signal": get_input_signals,  #vx steering
    "get_initial_state": lambda y_meas, u, theta: np.hstack((u[0]*u[0]*u[1]/theta[4]/2.65, y_meas[0])),
}

system, c0, theta_true, delta_theta = create_system(config_dyn)

mhe_conf = {
    "measurements_residual_r": np.diag([0.05, 1.0]),
    "state_prior_q0": np.diag([1.0, 1.0]),
    "noise_peanlty_w": np.eye(2) * 1e3,
    "fim_scaler": 0.2,
    "bounds_noise": [[-0.01, 0.01]] * 2,
    "bounds_state": [[-np.inf, np.inf]] * 2,
    "bounds_param": [[2, 25], [2, 25], [0.3, 0.65],[0.01, 0.25], [10, 20]],
}

mhe_params = create_mhe_params(mhe_conf, dt=0.02, mhe_horizont=200)
mhe_params.print()

dt: 0.02
mhe_horizont: 200
state_prior_q0: [[1. 0.]
 [0. 1.]]
noise_peanlty_w: [[1000.    0.]
 [   0. 1000.]]
measurements_residual_r: [[0.05 0.  ]
 [0.   1.  ]]
bounds_param: [[2, 25], [2, 25], [0.3, 0.65], [0.01, 0.25], [10, 20]]
bounds_state: [[-inf, inf], [-inf, inf]]
bounds_noise: [[-0.01, 0.01], [-0.01, 0.01]]
fim_scaler: 0.2
use_noise: 0


In [13]:
measured_batches = np.vstack((f_lat_ax(t), f_yaw_rate1(t))).T
def create_windows(t, measured_data, u, window_size, overlap):
    """
    Формирует скользящие окна с перекрытием для MHE.
    
    Параметры:
        t : np.ndarray, shape (N,)
        measured_data : np.ndarray, shape (N, n_obs)
        u : np.ndarray, shape (N, n_controls) или (N,) — может быть None, если управлений нет
        window_size : int, количество точек в окне
        overlap : int, количество перекрывающихся точек между соседними окнами
    
    Возвращает:
        t_windows : list of np.ndarray, каждый длины window_size
        u_windows : list of np.ndarray, каждый формы (window_size, n_controls) или (window_size,)
        meas_windows : list of np.ndarray, каждый формы (window_size, n_obs)
    """
    N = len(t)
    step = window_size - overlap
    if step <= 0:
        raise ValueError("overlap должен быть меньше window_size")
    
    starts = list(range(0, N - window_size + 1, step))  # только полные окна
    
    t_windows = []
    u_windows = []
    meas_windows = []
    
    for start in starts:
        idx = slice(start, start + window_size)
        t_windows.append(t[idx])
        meas_windows.append(measured_data[idx, :])
        if u is not None:
            u_windows.append(u[idx, ...])  # сохраняем размерность
        else:
            u_windows.append(None)  # или пропускаем
    
    return t_windows, u_windows, meas_windows




In [14]:
theta_true

array([ 2.88329956,  5.67880555,  0.48323357,  0.07280245, 12.47372793])

In [15]:
window_size = 200
overlap = 100
input_signals = np.vstack(get_input_signals(t)).T
t_windows, u_windows, meas_windows = create_windows(t, measured_batches, input_signals, window_size, overlap)

In [16]:
plot_mhe_data_windows(t_windows, u_windows, meas_windows, full_windows = None,
                      max_windows=5, state_idx=[0, 1])

<Figure size 1200x1500 with 10 Axes>

In [25]:
from acados_template import AcadosOcp
import os
class MyCogeGenerator(MheCogeGenerator):
    def __init__(self):
        super().__init__(system, mhe_params,  Path(os.getcwd())/'tmp_generated', 'dynamic_mhe')

    def modify_ocp_problem(self, ocp_mhe: AcadosOcp) -> AcadosOcp:
        ocp_mhe.solver_options.print_level = 1
        ocp_mhe.solver_options.nlp_solver_stats_level = 2

        # ocp_mhe.solver_options.integrator_type = 'IRK'
        # ocp_mhe.solver_options.sim_method_num_stages = 3
        # ocp_mhe.solver_options.sim_method_newton_tol = 1e-9
        # ocp_mhe.solver_options.sim_method_newton_iter = 10

        # # В настройках решателя
        # ocp_mhe.solver_options.nlp_solver_type = 'SQP'
        ocp_mhe.solver_options.nlp_solver_max_iter = 30
        # ocp_mhe.solver_options.levenberg_marquardt = 1e-6
        # ocp_mhe.solver_options.globalization = 'FIXED_STEP'
        # ocp_mhe.solver_options.globalization_fixed_step_length = 0.9
        # ocp_mhe.solver_options.nlp_solver_tol_stat = 1e-4   # временно ослабить
        # ocp_mhe.solver_options.nlp_solver_tol_eq = 1e-4
        # ocp_mhe.solver_options.nlp_solver_tol_ineq = 1e-4
        # ocp_mhe.solver_options.nlp_solver_tol_comp = 1e-4

        # ocp_mhe.solver_options.nlp_solver_max_iter =120
        # #ocp_mhe.solver_options.nlp_solver_tol_stat = 1e-3
        # ocp_mhe.solver_options.nlp_solver_type = 'SQP'
        # ocp_mhe.solver_options.nlp_solver_max_iter = 200
        # ocp_mhe.solver_options.nlp_solver_tol_stat = 1e-4
        # ocp_mhe.solver_options.nlp_solver_tol_eq = 1e-4
        # ocp_mhe.solver_options.nlp_solver_tol_ineq = 1e-4
        # ocp_mhe.solver_options.nlp_solver_tol_comp = 1e-4
        # ocp_mhe.solver_options.hessian_approx = 'GAUSS_NEWTON'
        # ocp_mhe.solver_options.globalization = 'FIXED_STEP'
        # ocp_mhe.solver_options.globalization_fixed_step_length = 0.9
        # ocp_mhe.solver_options.levenberg_marquardt = 1e-6
        # ocp_mhe.solver_options.qp_solver = 'PARTIAL_CONDENSING_HPIPM'
        # ocp_mhe.solver_options.hpipm_options = {
        #     'scale': 1,
        #     'scale_ux': 1,
        #     'iter_max': 1000,
        #     'tol': 1e-6,
        #     'reg_epsilon': 1e-6,
        #     'reg_epsilon_s': 1e-6,
        # }
        return ocp_mhe

generator = MyCogeGenerator()

#assert check_system_ok(system) == True

acados_solver_mhe = generator.generate_code()

Please export ACADOS_SOURCE_DIR to avoid this warning.
[ 2.    2.    0.3   0.01 10.  ]
[25.   25.    0.65  0.25 20.  ]
[2 3 4 5 6]
dynamic_mhe
5 2

got cost_type EXTERNAL for cost_type_0, cost_type, cost_type_e, hessian_approx: 'GAUSS_NEWTON'.
With this setting, acados will proceed computing the exact Hessian for the cost term and no Hessian contribution from constraints and dynamics.
If the external cost is a linear least squares cost, this coincides with the Gauss-Newton Hessian.
Note: There is also the option to use the external cost module with a numerical Hessian approximation (see `ext_cost_num_hess`).
OR the option to provide a symbolic custom Hessian approximation (see `cost_expr_ext_cost_custom_hess`).


got cost_type EXTERNAL for cost_type_0, cost_type, cost_type_e, hessian_approx: 'GAUSS_NEWTON'.
With this setting, acados will proceed computing the exact Hessian for the cost term and no Hessian contribution from constraints and dynamics.
If the external cost is a linear leas

In [18]:
def get_window(i):
    return t_windows[i], u_windows[i][:], meas_windows[i], None

acados_solver_mhe.reset()
initial_theta = theta_true
t, input_signal, measurements, _ = get_window(0)
x0_est = system.get_initial_state(measurements[0], input_signal[0], initial_theta)
reset_mhe_solver(generator.get_model(), 
                 acados_solver_mhe,
                 input_signal,
                 x0_est,
                 initial_theta,
                 window_size,
                 mhe_params.dt
                 )
print("initial_theta: ", initial_theta)
print("theta_true: ", theta_true)

create_step_function reset_traj
5
initial_theta:  [ 2.88329956  5.67880555  0.48323357  0.07280245 12.47372793]
theta_true:  [ 2.88329956  5.67880555  0.48323357  0.07280245 12.47372793]


In [19]:
mhe_model=generator.get_model()
mhe_model.system.nx

2

In [26]:


nx = generator.get_model().state_length
n_theta = generator.get_model().param_length
initial_std = np.array([2,2,0.1,0.1, 1])

initial_Sigma = np.eye(nx + n_theta)
initial_Sigma[:nx, :nx] *= 1e3          # большая неопределённость для состояний
initial_Sigma[nx:, nx:] = np.diag(initial_std ** 2)   # умеренная для параметров

results = run_mhe_estimation(
    mhe_model=generator.get_model(),
    acados_solver_factory=acados_solver_mhe,
    get_window_func=get_window,
    get_initial_state_func=system.get_initial_state,
    overlap_points=overlap,
    initial_theta=initial_theta,               # начальная оценка параметров
    mhe_params=mhe_params,
    num_windows=29,
    dt=mhe_params.dt,                                     # шаг дискретизации
    r_inv=mhe_params.measurements_residual_r,  # весовая матрица измерений
    Q_state_diag=1e-6,                         # шум процесса (состояния)
    initial_Sigma=initial_Sigma,               # начальная ковариация
    ridge_reg=1e-6,                            # регуляризация FIM
)

create_step_function ekf_step
5


MHE windows:   0%|          | 0/29 [00:00<?, ?window/s]2026-07-02 20:16:50,333 - WARNING - Window 0: acados status 1. Skip.
2026-07-02 20:16:50,341 - WARNING - Window 1: acados status 1. Skip.
2026-07-02 20:16:50,349 - WARNING - Window 2: acados status 1. Skip.
2026-07-02 20:16:50,356 - WARNING - Window 3: acados status 1. Skip.
2026-07-02 20:16:50,363 - WARNING - Window 4: acados status 1. Skip.
2026-07-02 20:16:50,371 - WARNING - Window 5: acados status 1. Skip.
2026-07-02 20:16:50,378 - WARNING - Window 6: acados status 1. Skip.
2026-07-02 20:16:50,386 - WARNING - Window 7: acados status 1. Skip.
2026-07-02 20:16:50,392 - WARNING - Window 8: acados status 1. Skip.
2026-07-02 20:16:50,400 - WARNING - Window 9: acados status 1. Skip.
2026-07-02 20:16:50,407 - WARNING - Window 10: acados status 1. Skip.
2026-07-02 20:16:50,414 - WARNING - Window 11: acados status 1. Skip.
2026-07-02 20:16:50,422 - WARNING - Window 12: acados status 1. Skip.
2026-07-02 20:16:50,429 - WARNING - Window 13

  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.
  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.
  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.
  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e

2026-07-02 20:16:50,524 - WARNING - Window 26: acados status 1. Skip.
2026-07-02 20:16:50,532 - WARNING - Window 27: acados status 1. Skip.
MHE windows:  97%|█████████▋| 28/29 [00:00<00:00, 136.03window/s]

  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.
  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.


2026-07-02 20:16:50,541 - WARNING - Window 28: acados status 1. Skip.
MHE windows: 100%|██████████| 29/29 [00:00<00:00, 134.88window/s]

  # it     res_stat       res_eq     res_ineq     res_comp   qp_stat   qp_iter  step_norm    lm_reg.     alpha   
     0          nan          nan   1.0000e+01   0.0000e+00         0         0   0.00e+00   0.00e+00  1.00e+00    
Stopped: NaN detected in iterate.
